In [ ]:
import numpy as np
import math

d = 50
n = 7

#upper_bound = ((n*d*(1-((d-1)/d)**2))/(1-n*d*((d-1)/d)**2))**(1/d)   #WRONG
#upper_bound = (d*n*d*(n*d+1)/2)**(1/d)  #WRONG
# upper_bound = math.comb(n*d, d)**(1/d)
# upper_bound = (n*math.comb(d, d))**(1/d)
lower_bound = (0.5+1/8*(3/n)**d+1/2*(n/3)**d)**(1/d) #PROBABILISTIC BOUND
# upper_bound = (d*(d+1))**(1/d)

# print(f"Upper bound: {upper_bound}")
print(f"Lower bound: {lower_bound}")


Lower bound: 2.301209643817838
0.9994150254340944


In [6]:
x = (2/0.0625)**(1/3)
d = 3
y = (2/7*math.ceil(3.25**d))**(1/(d-1))
print(x)
print(y)

# for i in range(112):
#     A = math.floor(i*2)%7
#     B = math.floor(i*2/x)%7
#     C = math.floor(i*2/x/x)%7
#     D = math.floor(i*2/x/x/x)%7
#     print(i, A, B, C, D)

# for i in range(112):
#     A = math.floor(i*2)%7
#     B = math.floor(i*2/x)%7
#     C = math.floor(i*2/x/x)%7
#     D = math.floor(i*2/x/x/x)%7
#     print(i, A, B, C, D)

3.1748021039363987
3.1622776601683795


In [7]:
from itertools import product
import math
import numpy as np
from tqdm import tqdm

def generate_shifts(d, shift_size):
    # Implementation for generating shifts
    # Generate all combinations of sets of integers of dimension d and largest integer shift_size-1, first integer can be larger than second etc
    shifts = np.array(list(product(range(shift_size), repeat=d)))
    return shifts

def count_adjacencies(point, points, p):
    count = 0
    #If for any dimension d , the absolute difference is larger than 1, not adjacent
    for other in points:
        is_adjacent = True
        for dim in range(len(point)):
            diff = abs(point[dim] - other[dim])
            if diff > 1 and diff < (p-1):
                is_adjacent = False
                break
        if is_adjacent:
            count += 1    
    return count

def return_adjacencies(points, p):
    adjacency_counts = []
    for point in points:
        count = count_adjacencies(point, points, p)
        adjacency_counts.append(count-1)  # Subtract 1 to not count itself
    return adjacency_counts

def create_independent_points(points, p):
    adjacency_counts = return_adjacencies(points, p)
    while (max(adjacency_counts) > 0):
        print(f"There are in total {np.count_nonzero(adjacency_counts)} nonzero adjacencies")
        max_index = adjacency_counts.index(max(adjacency_counts))
        points = np.delete(points, max_index, axis=0)
        adjacency_counts = return_adjacencies(points, p)
    return points

def generate_all_points(n, d):
    total_points = n ** d
    points = np.zeros((total_points, d), dtype=int)
    
    for i in range(total_points):
        for dim in range(d):
            points[i, dim] = (i // (n ** dim)) % n
    return points


def find_available_points(ind_set, other_set, d, p):
    available_points = []
    for point in other_set:
        is_available = True
        for ind_point in ind_set:
            # Check adjacency
            is_adjacent = True
            for dim in range(d):
                diff = abs(point[dim] - ind_point[dim])
                if diff > 1 and diff < (p-1):
                    is_adjacent = False
                    break
            if is_adjacent:
                is_available = False
                break
        if is_available:
            available_points.append(point)
    
    return np.array(available_points)

def extend_independent_set(ind_set, n, d):
    all_points = generate_all_points(n, d)
    other_set = np.array([point for point in all_points if point.tolist() not in ind_set.tolist()])
    available_points = find_available_points(ind_set, other_set, d, n)
    print(f"Found {len(available_points)} available points to extend the independent set.")
    while len(available_points) > 0:
        adjacency_counts = return_adjacencies(available_points, n)
        min_index = adjacency_counts.index(min(adjacency_counts))
        ind_set = np.vstack([ind_set, available_points[min_index]])
        available_points = np.delete(available_points, min_index, axis=0)
        available_points = find_available_points(ind_set, available_points, d, n)

    return ind_set

def shift_set(p, large_set):
    n = len(large_set)
    d = len(large_set[0])
    shift_size = math.ceil(2*(n-1)/p)
    print(f"Start generating shifts, shiftsize is {shift_size}")
    # shifts = generate_shifts(d, shift_size)
    # shifts = np.array([[40,123,40,123,40]])
    # print(f"Generated {len(shifts)} shifts.")
    # for shift in shifts:
    i = 0
    while i < 100:
        shift = np.random.randint(0, shift_size, size=d)
        # # if shift.tolist() == [40,14,40,14,40]:
        # #     print(((shifted_set - np.floor(((large_set + shift) % n) / shift_size))%p).tolist())
        shifted_set = np.floor(((large_set + shift) % n) / (shift_size/2))
        independent_points = create_independent_points(shifted_set, p)
        print(f"Created independent set of size {len(independent_points)} with q_array {q_array}")
        extended_set = extend_independent_set(independent_points, p, d)
        print(f"Extended to independent set of size {len(extended_set)} using the shift {shift}")
        i += 1

p = 7
q = 39
d = 6
n = 1146
print(f"Done importing")
q_array = np.array([q**i for i in range(d)])
large_set = (q_array * np.arange(n)[:, np.newaxis])%n
print(f"Generated large set of size {len(large_set)}")
shift_set(p, large_set)

Done importing
Generated large set of size 1146
Start generating shifts, shiftsize is 328


KeyboardInterrupt: 

In [6]:
import numpy as np
from tqdm import tqdm

from itertools import product
import math

def generate_shifts(d, shift_size):
    # Implementation for generating shifts
    # Generate all combinations of sets of integers of dimension d and largest integer shift_size-1, first integer can be larger than second etc
    shifts = np.array(list(product(range(shift_size), repeat=d)))
    return shifts

def check_adjacent(point1, point2, p, k):
    for dim in range(len(point1)):
        diff = abs(point1[dim] - point2[dim])
        if diff > (k-1) and diff < (p-(k-1)):
            return False
    return True

def count_adjacencies(point, points, p, k):
    count = 0
    #If for any dimension d , the absolute difference is larger than 1, not adjacent
    for other in points:
        if check_adjacent(point, other, p, k)==True:
            count+=1   
    return count

def return_adjacencies(points, p, k):
    adjacency_counts = []
    for point in points:
        count = count_adjacencies(point, points, p, k)
        adjacency_counts.append(count-1)  # Subtract 1 to not count itself
    return adjacency_counts

def create_independent_points(points, p, k):
    adjacency_counts = return_adjacencies(points, p, k)
    while (max(adjacency_counts) > 0):
        # max_index = adjacency_counts.index(max(adjacency_counts))
        # Set max_index random among the indices that are positive in adjacency_counts
        nonzero_indices = [i for i, count in enumerate(adjacency_counts) if count > 0]
        # max_index = np.random.choice(nonzero_indices)
        max_index = nonzero_indices[0]
        points = np.delete(points, nonzero_indices, axis=0)
        adjacency_counts = return_adjacencies(points, p, k)
    return points

def generate_all_points(n, d):
    total_points = n ** d
    points = np.zeros((total_points, d), dtype=int)
    
    for i in range(total_points):
        for dim in range(d):
            points[i, dim] = (i // (n ** dim)) % n
    return points


def find_available_points(ind_set, other_set, d, p, k):
    available_points = []
    for point in other_set:
        is_available = True
        for ind_point in ind_set:
            # Check adjacency
            if check_adjacent(ind_point, point, p, k):
                is_available = False
                break
        if is_available:
            available_points.append(point)
    
    return np.array(available_points)

def extend_independent_set(ind_set, n, d, k):
    all_points = generate_all_points(n, d)
    other_set = np.array([point for point in all_points if point.tolist() not in ind_set.tolist()])
    available_points = find_available_points(ind_set, other_set, d, n, k)
    print(f"Found {len(available_points)} available points to extend the independent set.")
    while len(available_points) > 0:
        adjacency_counts = return_adjacencies(available_points, n, k)
        min_index = adjacency_counts.index(min(adjacency_counts))
        ind_set = np.vstack([ind_set, available_points[min_index]])
        available_points = np.delete(available_points, min_index, axis=0)
        available_points = find_available_points(ind_set, available_points, d, n, k)

    return ind_set

def shift_set(p, large_set):
    n = len(large_set)
    d = len(large_set[0])
    shift_size = math.ceil(2*(n-1)/p)
    print(f"Start generating shifts, shiftsize is {shift_size}")
    # shifts = generate_shifts(d, shift_size)
    # shifts = np.array([[40,123,40,123,40]])
    # print(f"Generated {len(shifts)} shifts.")
    # for shift in shifts:
    i = 0
    while i < 1:
        # q_array = np.ones(5)
        # for j in range(4):
        #     q_array[j+1] = np.random.randint(q_array[j]+1, n-1-(3-j))
        # large_set = (q_array * np.arange(n)[:, np.newaxis])%n
        # shift = np.random.randint(0, shift_size, size=d)
        # # if shift.tolist() == [40,14,40,14,40]:
        # #     print(((shifted_set - np.floor(((large_set + shift) % n) / shift_size))%p).tolist())
        shift = np.array([40,124,40,123,40])
        shifted_set = np.floor(((large_set + shift) % n) / (shift_size/2))
        independent_points = create_independent_points(shifted_set, p, 2)
        print(f"Created independent set of size {len(independent_points)} with q_array {q_array}")
        extended_set = extend_independent_set(independent_points, p, d, 2)
        print(f"Extended to independent set of size {len(extended_set)} using the shift {shift}")
        i += 1
    return independent_points


def maximum_independent_set_exact(points, check_adjacency):
    """
    Exact maximum independent set via branch-and-bound with bitsets.

    Parameters
    ----------
    points : array-like of shape (N, d)
        Candidate vertices.
    check_adjacency : callable
        Function check_adjacency(p1, p2) -> bool where True means
        p1 and p2 are adjacent in the conflict graph (so they cannot both
        be in the independent set).

    Returns
    -------
    best_indices : list[int]
        Indices of vertices in a maximum independent set.
    best_points : np.ndarray
        The corresponding points.
    """
    points = np.asarray(points)
    n = len(points)

    if n == 0:
        return [], np.empty((0, 0), dtype=int)

    # Build adjacency bitmasks of the conflict graph.
    adj = [0] * n
    num_edges = 0
    for i in range(n):
        for j in range(i + 1, n):
            if check_adjacency(points[i], points[j]):
                adj[i] |= 1 << j
                adj[j] |= 1 << i
                num_edges += 1

    print(f"Graph has {n} nodes and {num_edges} edges")

    full_mask = (1 << n) - 1
    best_set = 0
    best_size = 0

    def popcount(x):
        return x.bit_count()

    def choose_vertex(candidates):
        # Branch first on a high-degree vertex for stronger pruning.
        x = candidates
        best_v = -1
        best_deg = -1
        while x:
            lsb = x & -x
            v = lsb.bit_length() - 1
            deg = popcount(adj[v] & candidates)
            if deg > best_deg:
                best_deg = deg
                best_v = v
            x ^= lsb
        return best_v

    def dfs(candidates, current_set, current_size):
        nonlocal best_set, best_size

        # Upper bound: even if we take all candidates, cannot beat current best.
        if current_size + popcount(candidates) <= best_size:
            return

        if candidates == 0:
            if current_size > best_size:
                best_size = current_size
                best_set = current_set
            return

        v = choose_vertex(candidates)
        bit_v = 1 << v

        # Include v: remove v and all its neighbors from candidates.
        dfs(candidates & ~adj[v] & ~bit_v, current_set | bit_v, current_size + 1)

        # Exclude v.
        dfs(candidates & ~bit_v, current_set, current_size)

    dfs(full_mask, 0, 0)

    best_indices = [i for i in range(n) if (best_set >> i) & 1]
    best_points = points[best_indices]
    return best_indices, best_points


# Example adapter for your existing signature check_adjacent(point1, point2, p, k)
# Replace p and k with your current values.
def make_check_adjacency(p, k):
    return lambda a, b: check_adjacent(a, b, p, k)

def experiment(p, large_set):
    n = len(large_set)
    d = len(large_set[0])
    k = 2
    shift_size = math.ceil(2*(n-1)/p)
    print(f"Start generating shifts, shiftsize is {shift_size}")
    # shifts = generate_shifts(d, shift_size)
    # shifts = np.array([[40,123,40,123,40]])
    # print(f"Generated {len(shifts)} shifts.")
    # for shift in shifts:
    all_points = generate_all_points(p, d)
    max_l = 0
    while max_l < 367:
        # q_array = np.ones(5)
        # for j in range(4):
        #     q_array[j+1] = np.random.randint(q_array[j]+1, n-1-(3-j))
        # large_set = (q_array * np.arange(n)[:, np.newaxis])%n
        # shift = np.random.randint(0, shift_size, size=d)
        # # if shift.tolist() == [40,14,40,14,40]:
        # #     print(((shifted_set - np.floor(((large_set + shift) % n) / shift_size))%p).tolist())
        shift = np.array([40,124,40,123,40])
        shifted_set = np.floor(((large_set + shift) % n) / (shift_size/2))
        independent_points = create_independent_points(shifted_set, p, 2)
        print(f"Created independent set of size {len(independent_points)} with q_array {q_array}")
        while len(independent_points) > 327:
            independent_points = np.delete(independent_points, np.random.randint(0, len(independent_points)), axis=0)
        other_set = np.array([point for point in all_points if point.tolist() not in independent_points.tolist()])
        available_points = find_available_points(independent_points, other_set, d, p, k)
        print(f"Found {len(available_points)} available points to extend the independent set.")
        check_adjacency = make_check_adjacency(p, k)
        best_idx, best_independent_set = maximum_independent_set_exact(available_points, check_adjacency)
        length = len(best_idx)+len(independent_points)
        print("Maximum independent set size:", length)
        if length > max_l:
            max_l = length
            max_set = np.vstack([best_independent_set, independent_points])

    return max_l, max_set

p = 7
q = 39
d = 5
n = 382
print(f"Done importing")
q_array = np.array([q**i for i in range(d)])
large_set = (q_array * np.arange(n)[:, np.newaxis])%n
max_l, max_set = experiment(p, large_set)

Done importing
Start generating shifts, shiftsize is 109
Created independent set of size 325 with q_array [      1      39    1521   59319 2313441]
Found 80 available points to extend the independent set.
Graph has 80 nodes and 141 edges


KeyboardInterrupt: 

518400


## Search for 3 permutations with distances in {4,5,6}

We fix the first permutation as:

\[
P_1 = (1,2,3,4,5,6,7,8,9,10)
\]

and require the other two permutations to also start with 1.

For each permutation, exactly 15 unordered number-pairs are at position distance 4, 5, or 6.
There are \(\binom{10}{2}=45\) unordered pairs total, so if 3 permutations satisfy the condition for every pair, the 3 sets of 15 covered pairs must partition all 45 pairs.

The code below:
1. Enumerates all permutations of 2..10 appended after 1.
2. Computes the 15-pair coverage mask for each permutation.
3. Searches for \(P_2, P_3\) so that masks of \(P_1, P_2, P_3\) are disjoint and their union is all pairs.

In [17]:
from itertools import permutations


def find_m_permutations(n, m, distances=None):
    """
    Search for m DISTINCT permutations of 1..n such that every unordered pair (a,b)
    is at an allowed position distance in at least one permutation.

    Symmetry reductions used:
    - First permutation is fixed as (1,2,...,n).
    - All other permutations start with 1.

    Parameters
    ----------
    n : int
        Numbers are 1..n.
    m : int
        Number of permutations to output.
    distances : set[int] | None
        Allowed distances. Defaults to {4, 5, 6}.

    Returns
    -------
    tuple[tuple[int, ...], ...] | None
        A tuple of m permutations if found, otherwise None.
    """
    if distances is None:
        distances = {4, 5, 6}
    distances = set(distances)

    if m < 1:
        return None

    # Index unordered pairs (a,b), a<b, into bit positions 0..C(n,2)-1.
    pair_to_bit = {}
    all_pairs = []
    bit = 0
    for a in range(1, n + 1):
        for b in range(a + 1, n + 1):
            pair_to_bit[(a, b)] = bit
            all_pairs.append((a, b))
            bit += 1

    num_pairs = len(all_pairs)
    full_mask = (1 << num_pairs) - 1

    def coverage_mask(perm):
        pos = {v: i for i, v in enumerate(perm)}
        mask = 0
        for a, b in all_pairs:
            if abs(pos[a] - pos[b]) in distances:
                mask |= 1 << pair_to_bit[(a, b)]
        return mask

    # Fixed first permutation.
    P1 = tuple(range(1, n + 1))
    M1 = coverage_mask(P1)

    if m == 1:
        return (P1,) if M1 == full_mask else None

    # Candidate permutations for remaining slots: must start with 1 and be distinct from P1.
    cand_perms = []
    cand_masks = []
    for tail in permutations(range(2, n + 1)):
        p = (1,) + tail
        if p == P1:
            continue
        cand_perms.append(p)
        cand_masks.append(coverage_mask(p))

    # For each pair-bit, store a big-int bitset of candidate indices covering it.
    pair_coverers = [0] * num_pairs
    for idx, mask in enumerate(cand_masks):
        x = mask
        while x:
            lsb = x & -x
            b = lsb.bit_length() - 1
            pair_coverers[b] |= 1 << idx
            x ^= lsb

    max_cover = max((mask.bit_count() for mask in cand_masks), default=0)

    target_len = m - 1

    def choose_hard_pair(uncovered_mask, used_idx_bits):
        """Pick an uncovered pair with the fewest available covering candidates."""
        best_bit = None
        best_avail = 0
        best_count = None

        x = uncovered_mask
        while x:
            lsb = x & -x
            b = lsb.bit_length() - 1
            avail = pair_coverers[b] & ~used_idx_bits
            count = avail.bit_count()
            if count == 0:
                return b, 0
            if best_count is None or count < best_count:
                best_count = count
                best_bit = b
                best_avail = avail
                if count == 1:
                    break
            x ^= lsb

        return best_bit, best_avail

    def dfs(covered_mask, used_idx_bits, chosen_indices):
        if len(chosen_indices) == target_len:
            return chosen_indices if covered_mask == full_mask else None

        uncovered = full_mask & ~covered_mask
        steps_left = target_len - len(chosen_indices)

        if uncovered == 0:
            # Already covers all pairs; pad with arbitrary unused candidates.
            padded = list(chosen_indices)
            idx = 0
            while len(padded) < target_len and idx < len(cand_perms):
                if ((used_idx_bits >> idx) & 1) == 0:
                    padded.append(idx)
                idx += 1
            return padded if len(padded) == target_len else None

        # Admissible pruning.
        if uncovered.bit_count() > steps_left * max_cover:
            return None

        _, avail = choose_hard_pair(uncovered, used_idx_bits)
        if avail == 0:
            return None

        # Try candidates covering the hard pair, high gain first.
        order = []
        x = avail
        while x:
            lsb = x & -x
            i = lsb.bit_length() - 1
            gain = (cand_masks[i] & uncovered).bit_count()
            order.append((gain, i))
            x ^= lsb
        order.sort(reverse=True)

        for _, i in order:
            new_covered = covered_mask | cand_masks[i]
            if new_covered == covered_mask:
                continue
            ans = dfs(new_covered, used_idx_bits | (1 << i), chosen_indices + [i])
            if ans is not None:
                return ans

        return None

    chosen = dfs(M1, 0, [])
    if chosen is None:
        return None

    solution = [P1] + [cand_perms[i] for i in chosen]

    # Final verification.
    pos_list = [{v: i for i, v in enumerate(P)} for P in solution]
    for a, b in all_pairs:
        if not any(abs(pos[a] - pos[b]) in distances for pos in pos_list):
            return None

    return tuple(solution)


# Example usage
n = 10
m = 3
D = {4, 5, 6}
result = find_m_permutations(n, m, D)

if result is None:
    print(f"No such {m} distinct permutations exist for n={n} under D={sorted(D)}.")
else:
    print(f"Found {m} valid distinct permutations for n={n}, D={sorted(D)}:")
    for i, perm in enumerate(result, 1):
        print(f"P{i} = {perm}")

No such 3 distinct permutations exist for n=10 under D=[4, 5, 6].


In [ ]:
#Poging om voor p=364 een mooie subgroup te maken maar met een generator omdat tja. 364,104, n=5
import numpy as np
X = np.array([1,4,11,35,104])
for i in range(364):
    print(i, (i*X)%364)


0 [0 0 0 0 0]
1 [  1   4  11  35 104]
2 [  2   8  22  70 208]
3 [  3  12  33 105 312]
4 [  4  16  44 140  52]
5 [  5  20  55 175 156]
6 [  6  24  66 210 260]
7 [  7  28  77 245   0]
8 [  8  32  88 280 104]
9 [  9  36  99 315 208]
10 [ 10  40 110 350 312]
11 [ 11  44 121  21  52]
12 [ 12  48 132  56 156]
13 [ 13  52 143  91 260]
14 [ 14  56 154 126   0]
15 [ 15  60 165 161 104]
16 [ 16  64 176 196 208]
17 [ 17  68 187 231 312]
18 [ 18  72 198 266  52]
19 [ 19  76 209 301 156]
20 [ 20  80 220 336 260]
21 [ 21  84 231   7   0]
22 [ 22  88 242  42 104]
23 [ 23  92 253  77 208]
24 [ 24  96 264 112 312]
25 [ 25 100 275 147  52]
26 [ 26 104 286 182 156]
27 [ 27 108 297 217 260]
28 [ 28 112 308 252   0]
29 [ 29 116 319 287 104]
30 [ 30 120 330 322 208]
31 [ 31 124 341 357 312]
32 [ 32 128 352  28  52]
33 [ 33 132 363  63 156]
34 [ 34 136  10  98 260]
35 [ 35 140  21 133   0]
36 [ 36 144  32 168 104]
37 [ 37 148  43 203 208]
38 [ 38 152  54 238 312]
39 [ 39 156  65 273  52]
40 [ 40 160  76 308 

In [9]:
from fractions import Fraction
import math

q = 2
alpha = 0.05
beta = 2
gamma = -1
n = 5
X = alpha*(q+gamma)**n+(q-gamma*alpha)**n

decimals = [(X+gamma*(alpha+1)*(q-gamma*alpha)**(n-1))/((q+gamma)*(X))]
for i in range(1,n-1):
    decimals.append(-gamma*alpha*(alpha+1)/X*(q+gamma)**(n-1)/(q-gamma*alpha)*(beta*(q+gamma)/(q-gamma*alpha))**(-i))
    decimals.append(-gamma*(alpha+1)/X*beta**(i)*(q+gamma)**(i-1)*(q-gamma*alpha)**(n-1-i))

def find_minimal_p(decimals):
    # Convert decimals to fractions in simplest form and get their denominators
    denominators = [Fraction(str(d)).denominator for d in decimals]
    
    # Compute the LCM of all denominators
    # (math.lcm supports multiple arguments in Python 3.9+)
    p = math.lcm(*denominators)
    
    return p

# Test
p = find_minimal_p(decimals)
print(f"The minimal p is: {p}")

for d in decimals:
    print(f"{d} * {p} = {d * p}")

detA = X/(alpha+1)

print(f"p/q = {p/q} and p/detA^1/n = {p/detA**(1/n)}")

The minimal p is: 2500000000000000000
0.4885112554237144 * 2500000000000000000 = 1.221278138559286e+18
0.0007240368093073488 * 2500000000000000000 = 1810092023268372.0
0.49901340934271776 * 2500000000000000000 = 1.2475335233567944e+18
0.0007421377295400324 * 2500000000000000000 = 1855344323850081.0
0.48684235057826136 * 2500000000000000000 = 1.2171058764456535e+18
0.0007606911727785332 * 2500000000000000000 = 1901727931946333.0
0.47496814690562084 * 2500000000000000000 = 1.187420367264052e+18
p/q = 1.25e+18 and p/detA^1/n = 1.2311306288788406e+18


In [18]:
from gekko import GEKKO

p = 7
q = 2
n = 7

m = GEKKO() # create GEKKO model
# create binary variables
a = m.Var(integer=False,lb=0,ub=1)
b = m.Var(integer=False,lb=0)
g = m.Var(integer=False)
z = m.Var(integer=True)
x1 = m.Var(integer=True)
x2 = m.Var(integer=True)
x3 = m.Var(integer=True)
x4 = m.Var(integer=True)
x5 = m.Var(integer=True)
x6 = m.Var(integer=True)
y1 = m.Var(integer=True)
y2 = m.Var(integer=True)
y3 = m.Var(integer=True)
y4 = m.Var(integer=True)
y5 = m.Var(integer=True)
y6 = m.Var(integer=True)
m.Equation(z == p*(a*(q+g)**n+(q-g*a)**n+g*(a+1)*(q-g*a)**(n-1))/(q+g)/(a*(q+g)**n+(q-g*a)**n))
m.Equation(x1 == -p*q*(a+1)/(a*(q+g)**n+(q-g*a)**n)*(q-g*a)**(n-1)/(q+g)*(beta*(q+g)/(q-g*a))**1)
m.Equation(x2 == -p*q*(a+1)/(a*(q+g)**n+(q-g*a)**n)*(q-g*a)**(n-1)/(q+g)*(beta*(q+g)/(q-g*a))**2)
m.Equation(x3 == -p*q*(a+1)/(a*(q+g)**n+(q-g*a)**n)*(q-g*a)**(n-1)/(q+g)*(beta*(q+g)/(q-g*a))**3)
m.Equation(x4 == -p*q*(a+1)/(a*(q+g)**n+(q-g*a)**n)*(q-g*a)**(n-1)/(q+g)*(beta*(q+g)/(q-g*a))**4)
m.Equation(x5 == -p*q*(a+1)/(a*(q+g)**n+(q-g*a)**n)*(q-g*a)**(n-1)/(q+g)*(beta*(q+g)/(q-g*a))**5)
m.Equation(x6 == -p*q*(a+1)/(a*(q+g)**n+(q-g*a)**n)*(q-g*a)**(n-1)/(q+g)*(beta*(q+g)/(q-g*a))**6)
m.Equation(y1 == -p*q*a*(a+1)/(a*(q+g)**n+(q-g*a)**n)*(q+g)**(n-1)/(q-g*a)*(beta*(q+g)/(q-g*a))**(-1))
m.Equation(y2 == -p*q*a*(a+1)/(a*(q+g)**n+(q-g*a)**n)*(q+g)**(n-1)/(q-g*a)*(beta*(q+g)/(q-g*a))**(-2))
m.Equation(y3 == -p*q*a*(a+1)/(a*(q+g)**n+(q-g*a)**n)*(q+g)**(n-1)/(q-g*a)*(beta*(q+g)/(q-g*a))**(-3))
m.Equation(y4 == -p*q*a*(a+1)/(a*(q+g)**n+(q-g*a)**n)*(q+g)**(n-1)/(q-g*a)*(beta*(q+g)/(q-g*a))**(-4))
m.Equation(y5 == -p*q*a*(a+1)/(a*(q+g)**n+(q-g*a)**n)*(q+g)**(n-1)/(q-g*a)*(beta*(q+g)/(q-g*a))**(-5))
m.Equation(y6 == -p*q*a*(a+1)/(a*(q+g)**n+(q-g*a)**n)*(q+g)**(n-1)/(q-g*a)*(beta*(q+g)/(q-g*a))**(-6))
m.Minimize((a*(q+g)**n+(q-g*a)**n)/(a+1)-q**n)
m.options.SOLVER = 3 # APOPT solver
m.solve()
# Print results
print(f"Optimal a: {a.value[0]}")
print(f"Optimal b: {b.value[0]}")
print(f"Optimal g: {g.value[0]}")

result = (a.value[0]+1)*p**n/(a.value[0]*(q+g.value[0])**n+(q-g.value[0]*a.value[0])**n)
print(f"Optimal result: {result} or better stated as {result**(1/n)}")

 ----------------------------------------------------------------
 APMonitor, Version 1.0.3
 APMonitor Optimization Suite
 ----------------------------------------------------------------
 
 
 --------- APM Model Size ------------
 Each time step contains
   Objects      :  0
   Constants    :  0
   Variables    :  16
   Intermediates:  0
   Connections  :  0
   Equations    :  14
   Residuals    :  14
 
 Number of state variables:    16
 Number of total equations: -  13
 Number of slack variables: -  0
 ---------------------------------------
 Degrees of freedom       :    3
 
 **********************************************
 Steady State Optimization with Interior Point Solver
 **********************************************
  
  
 Info: Exact Hessian

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).


In [6]:
import gurobipy as gp
from gurobipy import GRB

# Fixed parameter: n (e.g., n=3)
n = 5

# Create model
model = gp.Model("nonlinear_optimization")
model.setParam(GRB.Param.NonConvex, 2)  # Allow non-convex nonlinearities
model.setParam(GRB.Param.BestObjStop, (367.0)**(1/n))  # Stop when a feasible solution of at least 33 is found

# Variables
a = model.addVar(lb=1e-6, ub=1, name="a")
b = model.addVar(lb=1e-7, name="b")  # Strictly positive to avoid div by zero
g = model.addVar(lb=-GRB.INFINITY, name="g")  # Removed the positive bound on g
x = model.addVar(lb=1, name="x")

# Parameters p and q
p = 7 * x
q = 2 * x

# Precompute nonlinear terms
q_plus_g = q + g
q_minus_ga = q - g * a
term1 = a * (q_plus_g)**n
term2 = (q_minus_ga)**n
denominator = term1 + term2

# Objective: Maximize p / ((term1 + term2)/(a+1))^(1/n)
# NLExprs must be assigned to variables first before doing algebra
denom_var = model.addVar(lb=-GRB.INFINITY, name="denom_var")
model.addConstr(denom_var == denominator, name="denom_var_def")

obj_helper = model.addVar(lb=-GRB.INFINITY, name="obj_helper")
model.addConstr(obj_helper * denom_var == a + 1, name="obj_helper_def")

obj_val = model.addVar(lb=-GRB.INFINITY, name="obj_val")
model.addConstr(obj_val == p * (obj_helper)**(1/n), name="obj_val_def")
model.setObjective(obj_val, GRB.MAXIMIZE)

# --- Constraint 1: Must be integer ---
numerator1 = p * (term1 + term2 + g * (a + 1) * (q_minus_ga)**(n - 1))
denominator1 = q_plus_g * denominator

num1_var = model.addVar(lb=-GRB.INFINITY, name="num1_var")
den1_var = model.addVar(lb=-GRB.INFINITY, name="den1_var")
model.addConstr(num1_var == numerator1, name="num1_def")
model.addConstr(den1_var == denominator1, name="den1_def")

int_var1 = model.addVar(vtype=GRB.INTEGER, lb=-GRB.INFINITY, name="int_var1")
model.addConstr(int_var1 * den1_var == num1_var, name="constraint1")
model.addConstr(q_minus_ga >= 1e-7, name="avoid_zero_division")
model.addConstr(q_plus_g >= 1e-7, name="avoid_zero_division_2")

# --- Constraints for 0 < i < n-1 ---
for i in range(1, n):  # i from 1 to n-1
    # Algebraically simplified to remove division from inside exponent
    numerator2 = p * (-g * (a + 1)) * (q_minus_ga)**(n - 1 - i) * (b * q_plus_g)**i
    denominator2 = denominator * q_plus_g
    
    num2_var = model.addVar(lb=-GRB.INFINITY, name=f"num2_var_{i}")
    den2_var = model.addVar(lb=-GRB.INFINITY, name=f"den2_var_{i}")
    model.addConstr(num2_var == numerator2, name=f"num2_def_{i}")
    model.addConstr(den2_var == denominator2, name=f"den2_def_{i}")
    
    int_var2 = model.addVar(vtype=GRB.INTEGER, lb=-GRB.INFINITY, name=f"int_var2_{i}")
    model.addConstr(int_var2 * den2_var == num2_var, name=f"constraint2_{i}")

    # Algebraically simplified
    numerator3 = p * (-g * a * (a + 1)) * (q_plus_g)**(n - 1 - i) * (q_minus_ga)**i
    denominator3 = denominator * q_minus_ga * (b**i)
    
    num3_var = model.addVar(lb=-GRB.INFINITY, name=f"num3_var_{i}")
    den3_var = model.addVar(lb=-GRB.INFINITY, name=f"den3_var_{i}")
    model.addConstr(num3_var == numerator3, name=f"num3_def_{i}")
    model.addConstr(den3_var == denominator3, name=f"den3_def_{i}")

    int_var3 = model.addVar(vtype=GRB.INTEGER, lb=-GRB.INFINITY, name=f"int_var3_{i}")
    model.addConstr(int_var3 * den3_var == num3_var, name=f"constraint3_{i}")

# Solve
model.optimize()

# Print solution
if model.status == GRB.OPTIMAL:
    print(f"Optimal solution for n={n}:")
    print(f"a = {a.X}, b = {b.X}, g = {g.X}, x = {x.X}")
    print(f"Objective value: {model.ObjVal}")
elif model.status == GRB.USER_OBJ_LIMIT:
    print(f"Stopped on objective limit! Solution found for n={n}:")
    print(f"a = {a.X}, b = {b.X}, g = {g.X}, x = {x.X}")
    print(f"First integer constraint: {int_var1.X}")
    print(f"Objective value: {model.ObjVal}")
else:
    print("No optimal solution found.")

Set parameter NonConvex to value 2
Set parameter BestObjStop to value 3.2578659667835161e+00
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i5-8265U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
BestObjStop  3.2578659667835161e+00
NonConvex  2

Optimize a model with 1 rows, 34 columns and 2 nonzeros (Max)
Model fingerprint: 0xd63e0010
Model has 1 linear objective coefficients
Model has 11 quadratic constraints
Model has 20 general nonlinear constraints (133 nonlinear terms)
Variable types: 25 continuous, 9 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+00]
  QMatrix range    [1e+00, 1e+00]
  QLMatrix range   [1e+00, 2e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e-07, 1e+00]
  RHS range        [1e-07, 1e-07]
  QRHS range       [1e-07, 1e+00]
  NLCon coe range  [1e+00, 7e+00]

Adde

In [7]:
print(f"Objective value is {(a.X+1)*(7*x.X)**5/(a.X*(2*x.X+g.X)**5+(2*x.X-g.X*a.X)**5)}")
print(f"first integer constraints is: {7*x.X*(a.X*(2*x.X+g.X)**n+(2*x.X-g.X*a.X)**n+g.X*(a.X+1)*(2*x.X-g.X*a.X)**(n-1))/(2*x.X+g.X)/(a.X*(2*x.X+g.X)**n+(2*x.X-g.X*a.X)**n)}")

Objective value is 512.9352647853102
first integer constraints is: 3.479354239431844


In [14]:
print((273*273*273*273)%3438)

1521
